In [2]:
%pip install roboflow

from roboflow import Roboflow
rf = Roboflow(api_key="34aeVIOAX5XcJCubVAHR")
project = rf.workspace("nissha-medical").project("eagle-eyes-tbhgj")
version = project.version(18)
dataset = version.download("yolov8")


loading Roboflow workspace...
loading Roboflow project...


In [3]:
dataset_root_path = dataset.location
print(f"Automatically Detected Dataset Path: {dataset_root_path}")
print(f"Type of object: {type(dataset)}")

# You can now use this variable to build the config_path
# Assuming the Roboflow download structure (YOLOv8)
# The data.yaml file is typically one level up from the 'train' folder.
config_path = os.path.join(dataset_root_path, 'data.yaml')
print(f"YOLOv8 Config Path: {config_path}")

Automatically Detected Dataset Path: /content/eagle-eyes-18
Type of object: <class 'roboflow.core.dataset.Dataset'>
YOLOv8 Config Path: /content/eagle-eyes-18/data.yaml


In [4]:
%pip install protobuf==3.20.3 tensorboard ultralytics

In [6]:
import os
import glob
from ultralytics import YOLO
import yaml

def run_full_pipeline():
    """
    Trains, validates, and tests a YOLOv8 model. Then, it classifies
    test images based on Q-block count and calculates the
    accuracy of this classification logic against file-name-based ground truth.
    """
    # config_path = 'datasets/eagle-eyes-4/data.yaml'

    # --- Pre-run Checks ---
    if not os.path.exists(config_path):
        print(f"Error: Configuration file not found at '{config_path}'")
        return

    # --- 1. Load a Pre-trained YOLOv8 Model ---
    print("Loading pre-trained YOLOv8n model...")
    model = YOLO('yolov8n.pt')

    # --- 2. Train the Model on Your Custom Data ---
    print(f"Starting training with configuration: {config_path}")
    results = model.train(
        data=config_path,
        epochs=10,
        imgsz=640,
        batch=-1,
        name='yolov8n_qblock_detector'
    )

    print("--- Training complete! ---")

    # --- 3. Display Final Validation Metrics ---
    print("\nFinal Validation Metrics from Training:")
    print(f"  mAP50-95: {results.box.map:.4f}")
    print(f"  Precision: {results.box.mp:.4f}")
    print(f"  Recall: {results.box.mr:.4f}")

    # --- 4. Load the Best Model for Inference ---
    print("\nLoading the best model for inference...")
    results_directory = results.save_dir
    best_model_path = os.path.join(results_directory, 'weights/best.pt')
    if not os.path.exists(best_model_path):
        print(f"Error: Could not find best model at '{best_model_path}'.")
        return

    best_model = YOLO(best_model_path)
    print(f"Successfully loaded model from: {best_model_path}")

    # --- 5. Test Model and Apply Business Logic ---
    # print(f"\n--- Starting Test & QC Phase ---")

    # with open(config_path, 'r') as f:
    #     data_yaml = yaml.safe_load(f)

    # if 'test' not in data_yaml or not data_yaml['test']:
    #     print("Warning: 'test' path not found in your YAML file. Skipping test phase.")
    #     return

    # base_dir = os.path.dirname(os.path.abspath(config_path))

    # test_img_dir = base_dir + "/test/images"

    # print(f"BASE DIRECTORY:", base_dir)
    # print(f"TEST IMAGE DIRECTORY:" ,test_img_dir)

    # class_names_to_ids = {v: k for k, v in data_yaml['names'].items()}

    # # 2. Get the specific ID for the 'qblock' class
    # if 'qblock' not in class_names_to_ids:
    #     print("Error: The class 'qblock' was not found in the dataset configuration.")
    #     return

    # QBLOCK_CLASS_ID = class_names_to_ids['qblock']
    # print(f"Counting only detections with Class ID: {QBLOCK_CLASS_ID} ('qblock')")

    # if not os.path.exists(test_img_dir):
    #     print(f"Error: Test directory specified in YAML does not exist: '{test_img_dir}'")
        # return

    print(f"\n--- Starting Test & QC Phase ---")

    with open(config_path, 'r') as f:
        data_yaml = yaml.safe_load(f)

    if 'test' not in data_yaml or not data_yaml['test']:
        print("Warning: 'test' path not found in your YAML file. Skipping test phase.")
        return

    base_dir = os.path.dirname(os.path.abspath(config_path))

    # Use os.path.join for cross-platform compatibility (fixing the previous string concatenation)
    test_img_dir = os.path.join(base_dir, 'test', 'images')

    print(f"BASE DIRECTORY:", base_dir)
    print(f"TEST IMAGE DIRECTORY:" ,test_img_dir)

    # --- FIX: ROBUST CLASS MAPPING LOGIC ---
    names_data = data_yaml['names']

    if isinstance(names_data, dict):
        # Case 1: 'names' is a dictionary (e.g., {0: 'qblock', 1: 'smallqblock'})
        # We invert it to get {'qblock': 0}
        class_names_to_ids = {v: k for k, v in names_data.items()}
    elif isinstance(names_data, list):
        # Case 2: 'names' is a list (e.g., ['qblock', 'smallqblock', 'clutter'])
        # The index (i) is the class ID.
        class_names_to_ids = {name: i for i, name in enumerate(names_data)}
    else:
        print("Error: 'names' field in data.yaml is neither a list nor a dictionary.")
        return
    # --- END FIX ---

    # 2. Get the specific ID for the 'qblock' class
    if 'qblock' not in class_names_to_ids:
        print("Error: The class 'qblock' was not found in the dataset configuration. Check your YAML 'names' list/dict.")
        return

    QBLOCK_CLASS_ID = class_names_to_ids['qblock']
    print(f"Counting only detections with Class ID: {QBLOCK_CLASS_ID} ('qblock')")

    if not os.path.exists(test_img_dir):
        print(f"Error: Test directory specified in YAML does not exist: '{test_img_dir}'")
        return

    image_extensions = ['*.bmp', '*.jpg', '*.jpeg', '*.png']
    test_images = []
    for ext in image_extensions:
        test_images.extend(glob.glob(os.path.join(test_img_dir, ext)))

    if not test_images:
        print(f"Warning: No images found in the test directory '{test_img_dir}'. Skipping test phase.")
    else:
        print(f"Found {len(test_images)} images to test from path specified in YAML.")

        # --- QC LOGIC & ACCURACY CALCULATION ADDED HERE ---

        # 1. DEFINE YOUR "GOOD" STATE:
        EXPECTED_QBLOCK_COUNT = 14
        print(f"Applying QC Rule: Image is 'GOOD' if Q-block count is exactly {EXPECTED_QBLOCK_COUNT}.")

        # 2. Run prediction. (Code here remains the same)
        predict_results = best_model.predict(
            source=test_images,
            save=True,
            project="test_results",
            name="predictions_with_qc_accuracy",
            exist_ok=True,
            conf=0.5 # Set a confidence threshold
        )

        print("\n--- QC Classification Results ---")

        correct_predictions = 0
        total_images_processed = 0

        # 3. Loop through results and classify each image
        for image_path, result in zip(test_images, predict_results):
            image_name = os.path.basename(image_path)

            # --- Determine Ground Truth (from filename) ---
            ground_truth = "UNKNOWN"
            if "_OK" in image_name.upper():
                ground_truth = "GOOD"
            elif "_NG" in image_name.upper():
                ground_truth = "NOT_GOOD"

            # --- MODIFIED: Determine Model Prediction (ONLY COUNT 'qblock') ---

            # Get the tensor of class IDs for all detected boxes
            class_ids = result.boxes.cls.cpu() # Move to CPU if necessary for numpy/tensor ops

            # Count only the instances where the class ID matches the 'qblock' ID
            qblock_count = (class_ids == QBLOCK_CLASS_ID).sum().item()

            # Use the filtered count for the QC logic
            num_detections = qblock_count
            model_prediction = "GOOD" if num_detections == EXPECTED_QBLOCK_COUNT else "NOT_GOOD"

            # --- Score the Prediction ---
            if ground_truth == "UNKNOWN":
                print(f"Image: {image_name:<45} | GT: {ground_truth:<9} | Pred: {model_prediction:<9} ({num_detections} blocks) | Result: [SKIPPED]")
                continue

            total_images_processed += 1

            if model_prediction == ground_truth:
                correct_predictions += 1
                status = "[CORRECT]"
            else:
                status = "[INCORRECT]"

            print(f"Image: {image_name:<45} | GT: {ground_truth:<9} | Pred: {model_prediction:<9} ({num_detections} blocks) | Result: {status}")

        print("\n--- Testing Complete! ---")
        print("Predicted test images are saved in 'test_results/predictions_with_qc_accuracy'")

        # --- 4. Calculate and Print Final Accuracy ---
        if total_images_processed > 0:
            accuracy = (correct_predictions / total_images_processed) * 100
            print("\n--- Final QC Model Accuracy ---")
            print(f"Total Test Images Scored: {total_images_processed}")
            print(f"Correct Predictions:    {correct_predictions}")
            print(f"QC Model Accuracy:        {accuracy:.2f}%")
        else:
            print("\nNo scorable images (containing '_OK' or '_NG') were found in the test set.")


if __name__ == '__main__':
    run_full_pipeline()


Loading pre-trained YOLOv8n model...
Starting training with configuration: /content/eagle-eyes-18/data.yaml
Ultralytics 8.3.223 🚀 Python-3.12.12 torch-2.8.0+cu126 CUDA:0 (Tesla T4, 15095MiB)
engine/trainer: agnostic_nms=False, amp=True, augment=False, auto_augment=randaugment, batch=-1, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/eagle-eyes-18/data.yaml, degrees=0.0, deterministic=True, device=None, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=10, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8n.pt, momentum=0.937, mosaic=1.0, multi_scale=False, name=yolov8n_qblock_detector7, 